## 1. plip

In [16]:
import subprocess
from pathlib import Path

# list of PDB files
pdb_files = ["5ifj_abc_cut.pdb", "5ig7_abc_cut.pdb", "5ijk_acx_cut.pdb"]
base_outdir = Path("plip_results")
base_outdir.mkdir(exist_ok=True)

for pdb_path in pdb_files:
    # setup
    pdb_name = Path(pdb_path).stem 
    target_dir = base_outdir / pdb_name
    target_dir.mkdir(exist_ok=True)
    
    # run PLIP
    result = subprocess.run([
        "plip",
        "-f", pdb_path,
        "-o", str(target_dir),
        "--peptides", "C",  # use peptides arg, Chain C
        "-x",  # Generate XML report for data parsing
        "-y",  # Generate PyMOL .pse session file
    ], capture_output=True, text=True)
    # control
    if result.returncode == 0:
        print(f"Results saved to {target_dir}")
    else:
        print(f"Error processing {pdb_name}:")
        print(result.stderr)

Results saved to plip_results/5ifj_abc_cut
Results saved to plip_results/5ig7_abc_cut
Results saved to plip_results/5ijk_acx_cut


## 2. extract interaction data from report

In [17]:
import xml.etree.ElementTree as ET

def get_interacting_residues(xml_path, peptide_chain="C"):
    tree = ET.parse(xml_path)  # parse PLIP report
    root = tree.getroot()
    
    interacting_res = set()
    
    # iterate over all interaction groups
    for interaction_group in root.findall(".//interactions/*"):
        for contact in interaction_group:
            
            res_chain = contact.find("reschain")
            res_num = contact.find("resnr")
            res_type = contact.find("restype")
            
            if None in (res_chain, res_num, res_type):
                continue
            
            # exclude peptide
            if res_chain.text != peptide_chain:
                interacting_res.add(
                    (res_chain.text, int(res_num.text), res_type.text)
                )
                
    return sorted(interacting_res, key=lambda x: (x[0], x[1]))

In [18]:
# test
residues = get_interacting_residues("plip_results/5ifj_abc_cut/5ifj_abc_cut_report.xml")
print(residues) # (reschain, resnr, restype)

[('A', 38, 'ALA'), ('A', 52, 'TRP'), ('A', 64, 'SER'), ('A', 66, 'TYR'), ('A', 110, 'LYS'), ('A', 111, 'ASP'), ('B', 37, 'TYR'), ('B', 38, 'VAL'), ('B', 107, 'TRP'), ('B', 116, 'LEU')]


## 3. extract and map sequences

In [19]:
from Bio.PDB import PDBParser
from Bio.Seq import MutableSeq, Seq
from Bio.SeqRecord import SeqRecord
from Bio.SeqIO import write


# 3 to 1-letter code conversion
d3to1 = {
    'CYS': 'C', 'ASP': 'D', 'SER': 'S', 'GLN': 'Q', 'LYS': 'K',
    'ILE': 'I', 'PRO': 'P', 'THR': 'T', 'PHE': 'F', 'ASN': 'N',
    'GLY': 'G', 'HIS': 'H', 'LEU': 'L', 'ARG': 'R', 'TRP': 'W',
    'ALA': 'A', 'VAL': 'V', 'GLU': 'E', 'TYR': 'Y', 'MET': 'M'
}

def get_pdb_info(pdb_file):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("protein", pdb_file)

    chains_data = {} # stores aa sequences per chaain {chain_id: MutableSeq}
    res_maps = {}    # maps PDB residue numbers to positions in the seq {chain_id: {pdb_res_num: seq_index}}
    ordered_chains = []

    for model in structure:
        for chain in model:
            ordered_chains.append(chain.id)
            seq_list = []
            mapping = {}
            current_idx = 0
            for res in chain:
                resname = res.get_resname()
                if resname in d3to1:
                    seq_list.append(d3to1[resname])
                    mapping[str(res.id[1])] = current_idx  # maps PDB residues number to the sequence index
                    current_idx += 1
            chains_data[chain.id] = MutableSeq("".join(seq_list))
            res_maps[chain.id] = mapping
            
    return chains_data, res_maps, ordered_chains

In [20]:
test = get_pdb_info("5ifj_abc_cut.pdb")
print(test) 

({'A': MutableSeq('EVQLLESGGGLVQPGGSLRLSCAASGFTFSTYAMSWVRQAPGKGLEWVSGISGS...SSA'), 'B': MutableSeq('QLVLTQSPSASASLGASVRLTCTLSSGHSSYVIAWHQQQSEKGPRYLMKVNSDG...LSQ'), 'C': MutableSeq('PLQPEQPFP')}, {'A': {'1': 0, '2': 1, '3': 2, '4': 3, '5': 4, '6': 5, '7': 6, '8': 7, '9': 8, '11': 9, '12': 10, '13': 11, '14': 12, '15': 13, '16': 14, '17': 15, '18': 16, '19': 17, '20': 18, '21': 19, '22': 20, '23': 21, '24': 22, '25': 23, '26': 24, '27': 25, '28': 26, '29': 27, '30': 28, '35': 29, '36': 30, '37': 31, '38': 32, '39': 33, '40': 34, '41': 35, '42': 36, '43': 37, '44': 38, '45': 39, '46': 40, '47': 41, '48': 42, '49': 43, '50': 44, '51': 45, '52': 46, '53': 47, '54': 48, '55': 49, '56': 50, '57': 51, '58': 52, '59': 53, '62': 54, '63': 55, '64': 56, '65': 57, '66': 58, '67': 59, '68': 60, '69': 61, '70': 62, '71': 63, '72': 64, '74': 65, '75': 66, '76': 67, '77': 68, '78': 69, '79': 70, '80': 71, '81': 72, '82': 73, '83': 74, '84': 75, '85': 76, '86': 77, '87': 78, '88': 79, '89': 80, '90': 8

## 4. generate mutants

In [21]:
from pathlib import Path

def generate_mutant_fastas(pdb_file, interacting_res, output_fasta):
    pdb_name = Path(pdb_file).stem
    chains_data, res_maps, ordered_chains = get_pdb_info(pdb_file)  # use helper function
    mutant_records = []


    # Add original seq as wt
    wt_seq = ":".join(["".join(chains_data[cid]) for cid in ordered_chains])  # join seqs of chains with ":"
    mutant_records.append(SeqRecord(Seq(wt_seq), id=f"{pdb_name}_WT", description=""))

    for mut_chain, res_num, res_type in interacting_res:

        res_num = str(res_num) 

        if mut_chain in chains_data and res_num in res_maps[mut_chain]:
            idx = res_maps[mut_chain][res_num]
            temp_complex = {cid: list(chains_data[cid]) for cid in ordered_chains}
            
            expected_aa = d3to1.get(res_type.upper(), res_type) 
            current_aa = temp_complex[mut_chain][idx]
            
            if current_aa != expected_aa:
                print(f"Mismatch at {mut_chain}{res_num}: Expected {expected_aa}, found {current_aa}")  # skips positions, where there is a mismatch
                continue

            if current_aa == 'A': # skips positions already alanine
                continue 
            
            temp_complex[mut_chain][idx] = 'A'
            
            full_complex_seq = ":".join(["".join(temp_complex[cid]) for cid in ordered_chains])
            header = f"{pdb_name}_{mut_chain}_{expected_aa}{res_num}A"
            
            mutant_records.append(SeqRecord(Seq(full_complex_seq), id=header, description=""))

    
    # Global interface alanine mutant (check if ok.... and debug)
    temp_complex_all = {cid: list(chains_data[cid]) for cid in ordered_chains}
    mutated_positions = []

    for mut_chain, res_num, res_type in interacting_res:
        res_num = str(res_num)

        if mut_chain in chains_data and res_num in res_maps[mut_chain]:
            idx = res_maps[mut_chain][res_num]

            expected_aa = d3to1.get(res_type.upper(), res_type)
            current_aa = temp_complex_all[mut_chain][idx]

            if current_aa != expected_aa:
                print(f"[ALL_MUT] Mismatch at {mut_chain}{res_num}: Expected {expected_aa}, found {current_aa}")
                continue

            if current_aa == 'A':
                continue

            temp_complex_all[mut_chain][idx] = 'A'
            mutated_positions.append(f"{mut_chain}{expected_aa}{res_num}A")

    # Only add if at least one mutation happened
    if mutated_positions:
        full_seq_all = ":".join(["".join(temp_complex_all[cid]) for cid in ordered_chains])
        header = f"{pdb_name}_ALL_INTERFACE_A"
        mutant_records.append(SeqRecord(Seq(full_seq_all), id=header, description=""))

    with open(output_fasta, "w") as f:
        write(mutant_records, f, "fasta")
    print(f"Generated {len(mutant_records) - 1} mutants for {pdb_name}")  # -1 --> original seq 

In [22]:
pdb_files = ["5ifj_abc_cut.pdb", "5ig7_abc_cut.pdb", "5ijk_acx_cut.pdb"]

for pdb in pdb_files:
    # get the xml path
    pdb_stem = Path(pdb).stem
    xml_path = f"plip_results/{pdb_stem}/{pdb_stem}_report.xml"
    
    # extract residues
    res_list = get_interacting_residues(xml_path)
    
    # generate fasta
    fasta_out = f"{pdb_stem}_mutants.fasta"
    generate_mutant_fastas(pdb, res_list, fasta_out)

Mismatch at A111: Expected D, found C
[ALL_MUT] Mismatch at A111: Expected D, found C
Generated 9 mutants for 5ifj_abc_cut
Generated 9 mutants for 5ig7_abc_cut
Generated 13 mutants for 5ijk_acx_cut


In [24]:
from Bio.PDB import PDBParser

parser = PDBParser(QUIET=True)
structure = parser.get_structure("protein", "5ifj_abc_cut.pdb")

for model in structure:
    for chain in model:
        if chain.id == "A":
            for res in chain:
                hetflag, resseq, icode = res.id
                if resseq == 111:
                    print("Full residue ID:", res.id)
                    print("Residue name:", res.get_resname())

Full residue ID: (' ', 111, ' ')
Residue name: ASP
Full residue ID: (' ', 111, 'A')
Residue name: CYS


## create master fasta

In [10]:
from Bio import SeqIO
from pathlib import Path

# List the FASTA files to combine
fasta_to_combine = ["5ifj_abc_cut_mutants.fasta", "5ig7_abc_cut_mutants.fasta", "5ijk_acx_cut_mutants.fasta"]
master_output_name = "combined_mutants.fasta"

def create_master_fasta(file_list, output_name):
    all_records = []
    
    for fasta_file in file_list:
        if Path(fasta_file).exists():
            # Read the records from the individual file
            records = list(SeqIO.parse(fasta_file, "fasta"))
            all_records.extend(records)
            print(f"Added {len(records)} records from {fasta_file}")
        else:
            print(f"Warning: {fasta_file} not found. Skipping.")

    # Write all collected records to the master file
    with open(output_name, "w") as f:
        SeqIO.write(all_records, f, "fasta")
    
    print(f"\nMaster file created: {output_name} (Total records: {len(all_records)})")


create_master_fasta(fasta_to_combine, master_output_name)

Added 10 records from 5ifj_abc_cut_mutants.fasta
Added 10 records from 5ig7_abc_cut_mutants.fasta
Added 14 records from 5ijk_acx_cut_mutants.fasta

Master file created: combined_mutants.fasta (Total records: 34)


## sanity check

### check numbering scheme

In [11]:
# List of your target PDB files
pdb_files = ["5ifj_abc_cut.pdb", "5ig7_abc_cut.pdb", "5ijk_acx_cut.pdb"]

for pdb in pdb_files:
    print(f"{'-'*30}")
    print(pdb)
    print(f"{'-'*30}")
  
    
    # Get info for this specific PDB
    try:
        chains_data, res_maps, ordered_chains = get_pdb_info(pdb)
        
        # Check the mapping for the first 5 residues of the first chain (usually 'A')
        first_chain = ordered_chains[0]
        first_five = list(res_maps[first_chain].items())[:5]
        
        print(f"Mapping for Chain {first_chain}:")
        print("PDB Number -> Sequence Index")
        for pdb_num, seq_idx in first_five:
            print(f"  Residue {pdb_num} is at index {seq_idx}")
            
    except Exception as e:
        print(f"Error processing {pdb}: {e}")

------------------------------
5ifj_abc_cut.pdb
------------------------------
Mapping for Chain A:
PDB Number -> Sequence Index
  Residue 1 is at index 0
  Residue 2 is at index 1
  Residue 3 is at index 2
  Residue 4 is at index 3
  Residue 5 is at index 4
------------------------------
5ig7_abc_cut.pdb
------------------------------
Mapping for Chain A:
PDB Number -> Sequence Index
  Residue 1 is at index 0
  Residue 2 is at index 1
  Residue 3 is at index 2
  Residue 4 is at index 3
  Residue 5 is at index 4
------------------------------
5ijk_acx_cut.pdb
------------------------------
Mapping for Chain A:
PDB Number -> Sequence Index
  Residue 1 is at index 0
  Residue 2 is at index 1
  Residue 3 is at index 2
  Residue 4 is at index 3
  Residue 5 is at index 4


### MSA

In [12]:
import subprocess
from Bio import AlignIO

# setup dir
output_dir = Path("alignments")
output_dir.mkdir(exist_ok=True)

# List of mutant fastas
final_fastas = [
    "5ifj_abc_cut_mutants.fasta", 
    "5ig7_abc_cut_mutants.fasta", 
    "5ijk_acx_cut_mutants.fasta"
]

for f in final_fastas:
    out = output_dir / f.replace(".fasta", "_aligned.fasta")
    # Using Muscle v5 syntax
    subprocess.run(["muscle", "-align", f, "-output", out], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print(f"Aligned: {out}")

Aligned: alignments/5ifj_abc_cut_mutants_aligned.fasta
Aligned: alignments/5ig7_abc_cut_mutants_aligned.fasta
Aligned: alignments/5ijk_acx_cut_mutants_aligned.fasta


In [13]:
from bokeh.layouts import gridplot
from bokeh.models.glyphs import Text, Rect
from bokeh.models import ColumnDataSource, Plot, Grid, Range1d
from bokeh.plotting import figure, show
import numpy as np
from Bio import AlignIO

import panel as pn
import panel.widgets as pnw

pn.extension()


def get_colors(seqs):
    """Make colors for amino acids in sequence"""
    # Create a color dictionary for the 20 standard amino acids
    aa_colors = {
        'A': 'red',  # Alanine
        'C': 'orange',  # Cysteine
        'D': 'blue',  # Aspartic acid
        'E': 'green',  # Glutamic acid
        'F': 'purple',  # Phenylalanine
        'G': 'cyan',  # Glycine
        'H': 'pink',  # Histidine
        'I': 'lightgreen',  # Isoleucine
        'K': 'yellow',  # Lysine
        'L': 'lightblue',  # Leucine
        'M': 'darkgreen',  # Methionine
        'N': 'lightpink',  # Asparagine
        'P': 'grey',  # Proline
        'Q': 'lightgrey',  # Glutamine
        'R': 'darkblue',  # Arginine
        'S': 'darkcyan',  # Serine
        'T': 'lime',  # Threonine
        'V': 'magenta',  # Valine
        'W': 'brown',  # Tryptophan
        'Y': 'black',  # Tyrosine
        '-': 'white'  # Gap
    }

    # Flatten the sequence list into a single list of residues
    text = [i for s in list(seqs) for i in s]

    # Map the sequence letters to their corresponding colors
    colors = [aa_colors.get(i, 'white') for i in text]  # Default to white if unknown amino acid
    return colors


def view_alignment(aln, fontsize="9pt", plot_width=800):
    """Bokeh sequence alignment view for protein sequences"""

    # make sequence and id lists from the aln object
    seqs = [rec.seq for rec in (aln)]
    ids = [rec.id for rec in aln]
    text = [i for s in list(seqs) for i in s]
    colors = get_colors(seqs)

    N = len(seqs[0])  # Length of each sequence
    S = len(seqs)  # Number of sequences
    width = .4

    x = np.arange(1, N+1)
    y = np.arange(0, S, 1)

    # Create a 2D grid of coords from the 1D arrays
    xx, yy = np.meshgrid(x, y)

    # Flatten the arrays
    gx = xx.ravel()
    gy = yy.flatten()

    # Rect coordinates with offset for proper display
    recty = gy + .5
    h = 1 / S

    # Create the ColumnDataSource with all the arrays
    source = ColumnDataSource(dict(x=gx, y=gy, recty=recty, text=text, colors=colors, font_size=[fontsize]*len(text), font_name=["courier"]*len(text)))

    # Plot height (based on number of sequences)
    plot_height = len(seqs) * 15 + 50

    x_range = Range1d(0, N + 1, bounds='auto')
    viewlen = N/2

    # View range for close-up view
    view_range = (0, viewlen)
    tools = "xpan, xwheel_zoom, reset, save"

    # Entire sequence view (no text, with zoom)
    p = figure(title=None, width=plot_width, height=200,
               x_range=x_range, y_range=(0, S), tools=tools,
               min_border=0, toolbar_location='below')
    rects = Rect(x="x", y="recty",  width=1, height=1, fill_color="colors",
                 line_color=None, fill_alpha=0.6)
    p.add_glyph(source, rects)
    p.yaxis.visible = False
    p.grid.visible = False

    # Sequence text view with ability to scroll along x-axis
    p1 = figure(title=None, width=plot_width, height=plot_height,
                x_range=view_range, y_range=ids, tools="xpan,reset",
                min_border=0, toolbar_location='below')  # , lod_factor=1)

    # Pass font size and font name explicitly through ColumnDataSource
    glyph = Text(x="x", y="y", text="text", text_align='center', text_color="black",
                 text_font="font_name", text_font_size="font_size")  # Bind to column names in source
    rects = Rect(x="x", y="recty",  width=1, height=1, fill_color="colors",
                 line_color=None, fill_alpha=0.4)
    p1.add_glyph(source, glyph)
    p1.add_glyph(source, rects)

    p1.grid.visible = False
    p1.xaxis.major_label_text_font_style = "bold"
    p1.yaxis.minor_tick_line_width = 0
    p1.yaxis.major_tick_line_width = 0

    p = gridplot([[p], [p1]], toolbar_location='below')

    # Display the plot directly in the notebook
    return p

In [14]:
# Visualize 5ifj to check the fix
aln = AlignIO.read("alignments/5ifj_abc_cut_mutants_aligned.fasta", "fasta")
show(view_alignment(aln))

### check integrity / validate mutations

In [15]:
from pathlib import Path
from Bio import SeqIO
import sys

def audit_mutants(fasta_path, window=5, log_file=None):
    # setup the directory and full path
    log_dir = Path("log")
    log_dir.mkdir(exist_ok=True)
    
    log_path = log_dir / log_file if log_file else None

    records = list(SeqIO.parse(fasta_path, "fasta"))
    if not records:
        return

    wt_seq = str(records[0].seq)  # original seq as 1st
    
    # redirect stdout
    original_stdout = sys.stdout
    f = open(log_path, 'w') if log_path else None
    if f: 
        sys.stdout = f
    
    try:
        valid_count = 0
        for rec in records[1:]:
            mut_seq = str(rec.seq)
            diffs = [i for i in range(len(wt_seq)) if wt_seq[i] != mut_seq[i]]
            
            if len(diffs) == 1:
                idx = diffs[0]
                status = "VALID" if mut_seq[idx] == 'A' else f"ERROR ({mut_seq[idx]})"
                if mut_seq[idx] == 'A': valid_count += 1
                
                start, end = max(0, idx - window), min(len(wt_seq), idx + window + 1)
                context_wt  = wt_seq[start:idx] + "[" + wt_seq[idx] + "]" + wt_seq[idx+1:end]
                context_mut = mut_seq[start:idx] + "[" + mut_seq[idx] + "]" + mut_seq[idx+1:end]
                
                print(f"Mutation: {rec.id} | {status}")
                print(f"  WT Context:  {context_wt}")
                print(f"  Mut Context: {context_mut}\n")
            else:
                print(f"Mutation: {rec.id} | FAILED (Found {len(diffs)} diffs)\n")
                
        print(f"RESULT: Verified {valid_count}/{len(records)-1} mutants.\n")

    finally:
        # clean up and reset terminal output
        if f:
            sys.stdout = original_stdout
            f.close()
            print(f"Log saved to: {log_path}")

files_to_log = [
    "5ifj_abc_cut_mutants.fasta", 
    "5ig7_abc_cut_mutants.fasta", 
    "5ijk_acx_cut_mutants.fasta",
]

for fasta in files_to_log:
    log_name = fasta.replace(".fasta", "_log.txt")
    audit_mutants(fasta, log_file=log_name)

Log saved to: log/5ifj_abc_cut_mutants_log.txt
Log saved to: log/5ig7_abc_cut_mutants_log.txt
Log saved to: log/5ijk_acx_cut_mutants_log.txt
